# Train Forge 8B on Google Colab Pro

**Before you start**
1. Menu **Runtime → Change runtime type → A100 GPU** (or **L4 GPU**). Click **Save**.
2. Menu **Runtime → Run all**.
3. When asked, allow access to Google Drive (the finished model is saved there).

Takes about 1–2 hours. Keep this browser tab open.

**When it's done**
1. Download the `.gguf` file from Google Drive folder **forge**.
2. On your Mac, in the project folder: `bash cloud/install_on_mac.sh ~/Downloads/<file>.gguf`
3. Chat: `uv run chat.py --model forge-8b`
4. Here in Colab: **Runtime → Disconnect and delete runtime** (stops using compute units).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install Unsloth (commands from Unsloth's official Qwen3 Colab notebook)

In [ ]:
%%capture
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## 3. Connect Google Drive and download the project

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/forge
!rm -rf /content/forge && git clone --depth 1 https://github.com/obsdnx/local-llm-.git /content/forge
%cd /content/forge
!wc -l training/*.jsonl | tail -1

## 4. Train (the long step)
Shows the loss every 5 steps and tests on held-out examples every 20 steps. Lower `eval_loss` is better; the best checkpoint is kept automatically.

In [ ]:
!python cloud/train_8b.py

## 5. Save the results to Google Drive

In [ ]:
!find /content/forge -name "*.gguf" -exec cp -v {} /content/drive/MyDrive/forge/ \;
!cp -r /content/forge/forge-8b-lora /content/drive/MyDrive/forge/
!ls -lh /content/drive/MyDrive/forge